# Boterdrop Solver - Jupyter Setup

Local CAPTCHA solver (Turnstile, cf_clearance, reCAPTCHA v3, AWS WAF).
Runs as FastAPI server with Camoufox browser.

## Step 1: Install Dependencies

In [ ]:
import subprocess, sys

# Install system deps
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "xvfb", "libasound2"], check=True)

# Install Python deps
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "fastapi==0.95.2", "uvicorn", "camoufox[fetch]", "loguru", "psutil", "playwright"], check=True)

print("Done")

## Step 2: Fetch Camoufox Browser

In [ ]:
import subprocess, sys

# Fetch Camoufox browser (first time only)
subprocess.run([sys.executable, "-m", "camoufox", "fetch"], check=True)
print("Camoufox fetched")

## Step 3: Config

In [ ]:
import json

config = {
    "headless": True,       # True for VPS, False for local with display
    "thread": 3,             # Browser instances (more = faster, more RAM)
    "page_count": 2,         # Pages per instance
    "proxy_support": False,  # True if you have proxies
    "proxy_file": "proxies.txt",
    "host": "0.0.0.0",
    "port": 8000,
    "debug": False
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=4)

print(f"Config saved. Server will run on port {config['port']}")
print(f"Threads: {config['thread']}, Headless: {config['headless']}")

## Step 4: Start Solver Server (Background)

In [ ]:
import subprocess, time, requests

# Start server in background
proc = subprocess.Popen(
    [sys.executable, "api_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f"Server starting... PID: {proc.pid}")

# Wait for server to be ready
for i in range(30):
    time.sleep(2)
    try:
        r = requests.get("http://localhost:8000/", timeout=2)
        if r.status_code == 200:
            print(f"Server ready after {(i+1)*2}s")
            break
    except:
        pass
else:
    print("Server may not be ready yet, check logs")

## Step 5: Test Turnstile Solve

In [ ]:
import requests

# Test Turnstile solver
# Cloudflare dashboard signup sitekey
r = requests.get("http://localhost:8000/turnstile", params={
    "sitekey": "0x4AAAAAAAJel0iaAR3mgkjp",
    "url": "https://dash.cloudflare.com/sign-up"
}, timeout=120)

data = r.json()
print(f"Status: {r.status_code}")
print(f"Token: {data.get('token', 'N/A')[:50]}..." if data.get('token') else f"Error: {data}")

## Step 6: Stop Server

In [ ]:
# Stop server when done
proc.terminate()
proc.wait()
print("Server stopped")

## API Endpoints

| Endpoint | Method | Params | Description |
|---|---|---|---|
| `/turnstile` | GET | sitekey, url | Cloudflare Turnstile token |
| `/clearance` | GET | url | cf_clearance cookie |
| `/aws-token` | GET | url | AWS WAF token cookie |
| `/recaptchaV3` | GET | sitekey, url | reCAPTCHA v3 token |
| `/` | GET | - | Dashboard/status |